# KinetoCheck Per-Exercise Training on Google Colab

This notebook trains **one ST-GAT model per exercise** on IntelliRehab skeleton data using GPU acceleration.

**Features:**
- ✅ Per-exercise model training (9 exercises)
- ✅ GPU acceleration with CUDA + AMP (Automatic Mixed Precision)
- ✅ Early stopping & LR scheduling
- ✅ Callbacks system (checkpoint, early stop, reduce LR on plateau)
- ✅ tqdm progress bars
- ✅ Download trained weights for each exercise

**Steps:**
1. Upload `SkeletonData/Simplified/*.txt` files to Colab (or mount Google Drive)
2. Install dependencies
3. Select which exercises to train
4. Train models on GPU
5. Download `stgat_exercise_X_best.pt` weights

## 1. Setup & Dependencies

In [ ]:
# Check GPU availability
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA version: {torch.version.cuda}")
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("⚠ WARNING: No GPU detected. Training will be SLOW on CPU.")
    print("In Google Colab: Runtime → Change runtime type → T4 GPU")

In [ ]:
# Install torch-geometric and tqdm
!pip install torch-geometric tqdm -q
print("✓ Dependencies installed")

## 2. Upload Dataset

**Option A: Upload directly to Colab**
- Click the folder icon on the left sidebar
- Create a folder `SkeletonData/Simplified/`
- Upload all `.txt` files from your local `SkeletonData/Simplified/` folder

**Option B: Mount Google Drive (recommended for large datasets)**
- Upload your dataset to Google Drive first
- Run the cell below to mount Drive

In [ ]:
# Option B: Mount Google Drive (uncomment if using Drive)
# from google.colab import drive
# drive.mount('/content/drive')
# DATA_DIR = '/content/drive/MyDrive/SkeletonData/Simplified'  # adjust path

# Option A: Use uploaded files in Colab
DATA_DIR = 'SkeletonData/Simplified'

In [ ]:
# Verify dataset is accessible
import os
if os.path.exists(DATA_DIR):
    txt_files = [f for f in os.listdir(DATA_DIR) if f.endswith('.txt')]
    print(f"✓ Found {len(txt_files)} .txt files in {DATA_DIR}")
    if len(txt_files) > 0:
        print(f"  Sample: {txt_files[0]}")
else:
    print(f"✗ ERROR: Directory not found: {DATA_DIR}")
    print("  Please upload your dataset first (see instructions above)")

## 3. Configuration

In [ ]:
# Training configuration
class Config:
    # Exercise registry (IntelliRehab)
    EXERCISES = {
        0: "Shoulder Flexion",
        1: "Shoulder Abduction",
        2: "Shoulder Forward Extension",
        3: "Elbow Flexion",
        4: "Shoulder Horizontal Abduction",
        5: "Shoulder Rotation",
        6: "Forearm Pronation/Supination",
        7: "Wrist Flexion/Extension",
        8: "Hand to Mouth",
    }
    
    # Data
    NUM_KEYPOINTS = 25          # Kinect skeleton joints
    KEYPOINT_DIM = 3            # x, y, z coordinates
    NUM_CLASSES = 2             # correct / incorrect
    SEQUENCE_LENGTH = 120       # frames per sequence
    
    # Model architecture
    GAT_HIDDEN_DIM = 64
    GAT_NUM_HEADS = 4
    GAT_NUM_LAYERS = 3
    GAT_DROPOUT = 0.3
    
    # Training
    BATCH_SIZE = 32
    LEARNING_RATE = 1e-3
    WEIGHT_DECAY = 1e-4
    EPOCHS = 100
    
    # Optimization
    GRAD_CLIP_NORM = 1.0
    USE_AMP = True  # Automatic Mixed Precision
    PIN_MEMORY = True
    
    # LR Scheduler
    LR_SCHEDULER_FACTOR = 0.5
    LR_SCHEDULER_PATIENCE = 7
    LR_SCHEDULER_MIN_LR = 1e-6
    
    # Early stopping
    EARLY_STOPPING_PATIENCE = 15
    EARLY_STOPPING_MIN_DELTA = 1e-4
    
    # Device
    DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

config = Config()
print(f"Device: {config.DEVICE}")
print(f"AMP enabled: {config.USE_AMP and config.DEVICE == 'cuda'}")
print(f"Epochs: {config.EPOCHS}, Batch size: {config.BATCH_SIZE}")
print(f"Exercises: {list(config.EXERCISES.keys())}")

## 4. Data Preprocessing

In [ ]:
import numpy as np

class SkeletonPreprocessor:
    """Preprocess IntelliRehab skeleton data."""
    
    def __init__(self, seq_length=120):
        self.seq_length = seq_length
    
    def normalize(self, keypoints):
        """Z-score normalization."""
        mean = np.mean(keypoints)
        std = np.std(keypoints)
        if std > 0:
            keypoints = (keypoints - mean) / std
        return keypoints
    
    def pad_or_truncate(self, keypoints):
        """Resize sequence to target length using interpolation."""
        num_frames = keypoints.shape[0]
        
        if num_frames == self.seq_length:
            return keypoints
        
        if num_frames >= self.seq_length:
            # Sample frames uniformly
            indices = np.linspace(0, num_frames - 1, self.seq_length, dtype=int)
            return keypoints[indices]
        else:
            # Interpolate to upsample
            original_indices = np.linspace(0, 1, num_frames)
            target_indices = np.linspace(0, 1, self.seq_length)
            
            flat = keypoints.reshape(num_frames, -1)
            interpolated = np.zeros((self.seq_length, flat.shape[1]), dtype=keypoints.dtype)
            for feat_idx in range(flat.shape[1]):
                interpolated[:, feat_idx] = np.interp(target_indices, original_indices, flat[:, feat_idx])
            
            return interpolated.reshape(self.seq_length, *keypoints.shape[1:])
    
    def reshape_to_joints(self, keypoints):
        """Reshape (frames, 75) → (frames, 25, 3)."""
        if keypoints.ndim == 2:
            num_frames, num_features = keypoints.shape
            if num_features == 75:  # 25 joints × 3 coords
                return keypoints.reshape(num_frames, 25, 3)
        return keypoints
    
    def process(self, keypoints):
        """Full preprocessing pipeline."""
        keypoints = self.reshape_to_joints(keypoints)
        keypoints = self.pad_or_truncate(keypoints)
        keypoints = self.normalize(keypoints)
        return keypoints.astype(np.float32)

print("✓ Preprocessor defined")

## 5. Dataset Class

In [ ]:
from torch.utils.data import Dataset

class SkeletonDataset(Dataset):
    """Load IntelliRehab .txt files with per-exercise filtering."""
    
    def __init__(self, data_dir, exercise_id=None, seq_length=120):
        """
        Args:
            data_dir: Path to Simplified/ folder
            exercise_id: Filter for specific exercise (0-8), or None for all
            seq_length: Target sequence length
        """
        self.preprocessor = SkeletonPreprocessor(seq_length)
        self.samples = []
        self.labels = []
        self.exercise_ids = []
        
        for fname in os.listdir(data_dir):
            if fname.endswith('.txt'):
                try:
                    # Parse: SubjectID_DateID_GestureID_RepNum_CorrectLabel_Position.txt
                    pieces = fname.replace('.txt', '').split('_')
                    if len(pieces) >= 5:
                        gesture_id = int(pieces[2])  # Exercise ID
                        correct_label_raw = int(pieces[4])
                        label = correct_label_raw - 1  # 1→0 (correct), 2→1 (incorrect)
                        
                        # Skip poorly executed (label=2)
                        if label == 2:
                            continue
                        
                        # Filter by exercise if specified
                        if exercise_id is not None and gesture_id != exercise_id:
                            continue
                        
                        self.samples.append(os.path.join(data_dir, fname))
                        self.labels.append(label)
                        self.exercise_ids.append(gesture_id)
                except (ValueError, IndexError):
                    continue
    
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        raw = np.loadtxt(self.samples[idx], delimiter=',', dtype=np.float32)
        processed = self.preprocessor.process(raw)
        x = torch.tensor(processed, dtype=torch.float32)
        y = torch.tensor(self.labels[idx], dtype=torch.long)
        return x, y
    
    def label_distribution(self):
        from collections import Counter
        return Counter(self.labels)

print("✓ SkeletonDataset (per-exercise) defined")

## 6. ST-GAT Model

In [ ]:
from dataclasses import dataclass, field
from typing import Dict, Any, List, Optional

@dataclass
class TrainingMetrics:
    """Snapshot of training state."""
    epoch: int = 0
    total_epochs: int = 0
    train_loss: float = 0.0
    train_acc: float = 0.0
    val_loss: float = 0.0
    val_acc: float = 0.0
    lr: float = 0.0
    epoch_time: float = 0.0

class TrainingCallback:
    """Base callback class."""
    def on_epoch_end(self, metrics: TrainingMetrics) -> None:
        pass
    
    @property
    def should_stop(self) -> bool:
        return False

class EarlyStopping(TrainingCallback):
    """Stop training when metric stops improving."""
    def __init__(self, patience=10, min_delta=1e-4, monitor="val_loss", mode="min"):
        self.patience = patience
        self.min_delta = min_delta
        self.monitor = monitor
        self.mode = mode
        self.best = None
        self.counter = 0
        self._stop = False
    
    def _is_improvement(self, current):
        if self.best is None:
            return True
        if self.mode == "min":
            return current < self.best - self.min_delta
        return current > self.best + self.min_delta
    
    def on_epoch_end(self, metrics):
        current = getattr(metrics, self.monitor)
        if self._is_improvement(current):
            self.best = current
            self.counter = 0
        else:
            self.counter += 1
            if self.counter >= self.patience:
                print(f"  ⏹ Early stopping: no improvement for {self.patience} epochs (best={self.best:.4f})")
                self._stop = True
    
    @property
    def should_stop(self):
        return self._stop

class ModelCheckpoint(TrainingCallback):
    """Save best model."""
    def __init__(self, save_path, model, monitor="val_acc", mode="max"):
        self.save_path = save_path
        self.model = model
        self.monitor = monitor
        self.mode = mode
        self.best = None
    
    def _is_improvement(self, current):
        if self.best is None:
            return True
        return current < self.best if self.mode == "min" else current > self.best
    
    def on_epoch_end(self, metrics):
        current = getattr(metrics, self.monitor)
        if self._is_improvement(current):
            self.best = current
            torch.save(self.model.state_dict(), self.save_path)
            print(f"  💾 Checkpoint: {self.monitor}={current:.4f} → {self.save_path}")

class ReduceLROnPlateauCallback(TrainingCallback):
    """Reduce LR on plateau."""
    def __init__(self, optimizer, monitor="val_loss", factor=0.5, patience=5, min_lr=1e-6):
        self.monitor = monitor
        self.scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer,
            mode="min" if "loss" in monitor else "max",
            factor=factor,
            patience=patience,
            min_lr=min_lr,
        )
    
    def on_epoch_end(self, metrics):
        current = getattr(metrics, self.monitor)
        old_lr = self.scheduler.optimizer.param_groups[0]['lr']
        self.scheduler.step(current)
        new_lr = self.scheduler.optimizer.param_groups[0]['lr']
        if new_lr < old_lr:
            print(f"  📉 LR reduced: {old_lr:.2e} → {new_lr:.2e}")

class CallbackRunner:
    """Manages multiple callbacks."""
    def __init__(self, callbacks=None):
        self.callbacks = callbacks or []
    
    def add(self, cb):
        self.callbacks.append(cb)
    
    def on_epoch_end(self, metrics):
        for cb in self.callbacks:
            cb.on_epoch_end(metrics)
    
    @property
    def should_stop(self):
        return any(cb.should_stop for cb in self.callbacks)

print("✓ Callbacks system defined")

In [ ]:
## 6B. ST-GAT Model

## 6A. Callbacks System

In [ ]:
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GATConv
from torch_geometric.data import Data, Batch

# Kinect skeleton edges (25 joints)
KINECT_EDGES = [
    (0, 1), (1, 20), (20, 2), (2, 3),                        # spine + head
    (20, 4), (4, 5), (5, 6), (6, 7), (7, 21), (7, 22),      # left arm
    (20, 8), (8, 9), (9, 10), (10, 11), (11, 23), (11, 24), # right arm
    (0, 12), (12, 13), (13, 14), (14, 15),                   # left leg
    (0, 16), (16, 17), (17, 18), (18, 19),                   # right leg
]

def build_edge_index(edges, num_nodes):
    """Build bidirectional edge_index with self-loops."""
    src = [e[0] for e in edges] + [e[1] for e in edges]
    dst = [e[1] for e in edges] + [e[0] for e in edges]
    src += list(range(num_nodes))
    dst += list(range(num_nodes))
    return torch.tensor([src, dst], dtype=torch.long)

class TemporalConvBlock(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size=3):
        super().__init__()
        self.conv = nn.Conv1d(in_channels, out_channels, kernel_size=kernel_size, padding=kernel_size // 2)
        self.bn = nn.BatchNorm1d(out_channels)
    
    def forward(self, x):
        return F.relu(self.bn(self.conv(x)))

class SpatialGATBlock(nn.Module):
    def __init__(self, in_channels, out_channels, heads=4, dropout=0.3):
        super().__init__()
        self.gat = GATConv(in_channels, out_channels // heads, heads=heads, dropout=dropout, concat=True)
        self.bn = nn.BatchNorm1d(out_channels)
    
    def forward(self, x, edge_index):
        return F.relu(self.bn(self.gat(x, edge_index)))

class STGATNetwork(nn.Module):
    """Spatial-Temporal Graph Attention Network."""
    
    def __init__(self, num_keypoints=25, keypoint_dim=3, hidden_dim=64, 
                 num_classes=2, num_layers=3, num_heads=4, seq_length=120, dropout=0.3):
        super().__init__()
        self.num_keypoints = num_keypoints
        self.hidden_dim = hidden_dim
        self.seq_length = seq_length
        
        self.input_proj = nn.Linear(keypoint_dim, hidden_dim)
        
        self.spatial_blocks = nn.ModuleList([
            SpatialGATBlock(hidden_dim, hidden_dim, heads=num_heads, dropout=dropout)
            for _ in range(num_layers)
        ])
        
        self.temporal_blocks = nn.ModuleList([
            TemporalConvBlock(hidden_dim, hidden_dim)
            for _ in range(num_layers)
        ])
        
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, num_classes)
        )
        
        self.register_buffer('edge_index', build_edge_index(KINECT_EDGES, num_keypoints))
    
    def forward(self, x):
        B, T, K, D = x.shape
        x = self.input_proj(x)  # (B, T, K, H)
        
        for spatial_blk, temporal_blk in zip(self.spatial_blocks, self.temporal_blocks):
            # Spatial GAT
            x_flat = x.reshape(B * T, K, self.hidden_dim)
            graphs = [Data(x=x_flat[i], edge_index=self.edge_index) for i in range(B * T)]
            batch = Batch.from_data_list(graphs)
            spatial_out = spatial_blk(batch.x, batch.edge_index)
            x = spatial_out.reshape(B, T, K, self.hidden_dim)
            
            # Temporal Conv
            x_perm = x.permute(0, 2, 3, 1).reshape(B * K, self.hidden_dim, T)
            temporal_out = temporal_blk(x_perm)
            x = temporal_out.reshape(B, K, self.hidden_dim, T).permute(0, 3, 1, 2)
        
        x = x.mean(dim=[1, 2])  # Global pooling
        return self.classifier(x)

print("✓ ST-GAT model architecture defined")

## 7. Select Exercises to Train

Choose which exercises to train (default: all 9)

In [ ]:
# Select which exercises to train (0-8)
# To train all: EXERCISES_TO_TRAIN = list(range(9))
# To train specific ones: EXERCISES_TO_TRAIN = [0, 3, 8]

EXERCISES_TO_TRAIN = list(range(9))  # Train all exercises

print(f"Will train {len(EXERCISES_TO_TRAIN)} exercises:")
for ex_id in EXERCISES_TO_TRAIN:
    print(f"  [{ex_id}] {config.EXERCISES.get(ex_id, f'Exercise {ex_id}')}")

## 8. Per-Exercise Training Function

In [ ]:
from torch.utils.data import DataLoader, random_split
from tqdm import tqdm
import time

def train_exercise(exercise_id, data_dir, config):
    """Train one model for a single exercise."""
    exercise_name = config.EXERCISES.get(exercise_id, f"Exercise {exercise_id}")
    print(f"\n{'═' * 60}")
    print(f"  Exercise {exercise_id}: {exercise_name}")
    print(f"{'═' * 60}")
    
    # Load dataset for this exercise
    dataset = SkeletonDataset(data_dir, exercise_id=exercise_id, seq_length=config.SEQUENCE_LENGTH)
    if len(dataset) == 0:
        print(f"  ⚠ No samples for exercise {exercise_id} — skipping.")
        return 0.0
    
    # Show distribution
    dist = dataset.label_distribution()
    total = len(dataset)
    for label, count in sorted(dist.items()):
        tag = "correct" if label == 0 else "incorrect"
        print(f"  {tag}: {count} ({100.0 * count / total:.1f}%)")
    
    # Split
    train_size = int(0.8 * total)
    val_size = total - train_size
    train_set, val_set = random_split(dataset, [train_size, val_size])
    
    use_cuda = config.DEVICE == 'cuda'
    train_loader = DataLoader(
        train_set, 
        batch_size=config.BATCH_SIZE, 
        shuffle=True, 
        num_workers=2,
        pin_memory=config.PIN_MEMORY and use_cuda
    )
    val_loader = DataLoader(
        val_set, 
        batch_size=config.BATCH_SIZE, 
        num_workers=2,
        pin_memory=config.PIN_MEMORY and use_cuda
    )
    
    print(f"  Train: {train_size}  Val: {val_size}")
    
    # Build model
    model = STGATNetwork(
        num_keypoints=config.NUM_KEYPOINTS,
        keypoint_dim=config.KEYPOINT_DIM,
        hidden_dim=config.GAT_HIDDEN_DIM,
        num_classes=config.NUM_CLASSES,
        num_layers=config.GAT_NUM_LAYERS,
        num_heads=config.GAT_NUM_HEADS,
        seq_length=config.SEQUENCE_LENGTH,
        dropout=config.GAT_DROPOUT
    ).to(config.DEVICE)
    
    total_params = sum(p.numel() for p in model.parameters())
    print(f"  Model: ST-GAT ({total_params:,} params)")
    print(f"  Device: {config.DEVICE}   AMP: {config.USE_AMP and use_cuda}")
    
    # Optimizer & loss
    optimizer = torch.optim.Adam(
        model.parameters(), 
        lr=config.LEARNING_RATE,
        weight_decay=config.WEIGHT_DECAY
    )
    criterion = nn.CrossEntropyLoss()
    
    # AMP scaler
    use_amp = config.USE_AMP and use_cuda
    scaler = torch.amp.GradScaler('cuda', enabled=use_amp)
    
    # Callbacks
    save_path = f'stgat_exercise_{exercise_id}_best.pt'
    callbacks = CallbackRunner()
    callbacks.add(EarlyStopping(
        patience=config.EARLY_STOPPING_PATIENCE,
        min_delta=config.EARLY_STOPPING_MIN_DELTA,
        monitor="val_loss",
        mode="min"
    ))
    callbacks.add(ModelCheckpoint(save_path, model, monitor="val_acc", mode="max"))
    callbacks.add(ReduceLROnPlateauCallback(
        optimizer,
        monitor="val_loss",
        factor=config.LR_SCHEDULER_FACTOR,
        patience=config.LR_SCHEDULER_PATIENCE,
        min_lr=config.LR_SCHEDULER_MIN_LR
    ))
    
    print(f"  Weights → {save_path}\n")
    
    # Training loop
    total_start = time.time()
    
    for epoch in tqdm(range(config.EPOCHS), desc="  Epochs", unit="ep"):
        epoch_start = time.time()
        
        # ── Train ──
        model.train()
        train_loss, train_acc = 0.0, 0.0
        
        for batch in tqdm(train_loader, desc="  train", leave=False):
            x, y = batch
            x = x.to(config.DEVICE, non_blocking=True)
            y = y.to(config.DEVICE, non_blocking=True)
            
            optimizer.zero_grad(set_to_none=True)
            
            with torch.amp.autocast(config.DEVICE, enabled=use_amp):
                outputs = model(x)
                loss = criterion(outputs, y)
            
            scaler.scale(loss).backward()
            
            if config.GRAD_CLIP_NORM > 0:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), config.GRAD_CLIP_NORM)
            
            scaler.step(optimizer)
            scaler.update()
            
            train_loss += loss.item()
            train_acc += (outputs.argmax(dim=-1) == y).float().mean().item()
        
        train_loss /= len(train_loader)
        train_acc /= len(train_loader)
        
        # ── Validate ──
        model.eval()
        val_loss, val_acc = 0.0, 0.0
        
        with torch.no_grad():
            for batch in tqdm(val_loader, desc="  val  ", leave=False):
                x, y = batch
                x = x.to(config.DEVICE, non_blocking=True)
                y = y.to(config.DEVICE, non_blocking=True)
                
                with torch.amp.autocast(config.DEVICE, enabled=use_amp):
                    outputs = model(x)
                    loss = criterion(outputs, y)
                
                val_loss += loss.item()
                val_acc += (outputs.argmax(dim=-1) == y).float().mean().item()
        
        val_loss /= len(val_loader)
        val_acc /= len(val_loader)
        
        # Metrics
        epoch_time = time.time() - epoch_start
        current_lr = optimizer.param_groups[0]['lr']
        
        metrics = TrainingMetrics(
            epoch=epoch + 1,
            total_epochs=config.EPOCHS,
            train_loss=train_loss,
            train_acc=train_acc,
            val_loss=val_loss,
            val_acc=val_acc,
            lr=current_lr,
            epoch_time=epoch_time
        )
        
        callbacks.on_epoch_end(metrics)
        
        if callbacks.should_stop:
            break
    
    total_time = time.time() - total_start
    
    # Get best accuracy from checkpoint callback
    ckpt = next((c for c in callbacks.callbacks if isinstance(c, ModelCheckpoint)), None)
    best_val_acc = ckpt.best if ckpt and ckpt.best is not None else 0.0
    
    print(f"\n  Exercise {exercise_id} done in {total_time / 60:.1f} min — best val_acc={best_val_acc:.4f}")
    return best_val_acc

print("✓ Training function defined")

## 9. Run Training for Selected Exercises

In [ ]:
results = {}

print(f"Training {len(EXERCISES_TO_TRAIN)} exercises on {config.DEVICE}...\n")
overall_start = time.time()

try:
    for ex_id in EXERCISES_TO_TRAIN:
        results[ex_id] = train_exercise(ex_id, DATA_DIR, config)
except KeyboardInterrupt:
    print("\n\nTraining interrupted by user.")

overall_time = time.time() - overall_start

# Summary
print(f"\n{'═' * 60}")
print("  Training Summary")
print(f"{'═' * 60}")
for ex_id, acc in results.items():
    name = config.EXERCISES.get(ex_id, f"Exercise {ex_id}")
    status = f"val_acc={acc:.4f}" if acc > 0 else "SKIPPED"
    print(f"  [{ex_id}] {name:35s} {status}")
print(f"\nTotal time: {overall_time / 60:.1f} minutes")
print(f"{'═' * 60}")

## 10. Download Trained Models

Download the trained weights for each exercise:
- Click the folder icon on the left sidebar
- Find files: `stgat_exercise_0_best.pt`, `stgat_exercise_1_best.pt`, etc.
- Right-click and select "Download"
- Place them in your local `Backend/weights/` directory

In [ ]:
# List all trained model files
import os
import glob

model_files = glob.glob('stgat_exercise_*_best.pt')
model_files.sort()

if model_files:
    print(f"✓ Found {len(model_files)} trained model(s):\n")
    total_size = 0
    for fname in model_files:
        size_mb = os.path.getsize(fname) / (1024 * 1024)
        total_size += size_mb
        ex_id = fname.split('_')[2]
        ex_name = config.EXERCISES.get(int(ex_id), f"Exercise {ex_id}")
        acc = results.get(int(ex_id), 0.0)
        print(f"  {fname:30s} ({size_mb:.2f} MB) — {ex_name} (acc={acc:.4f})")
    print(f"\nTotal size: {total_size:.2f} MB")
    print("\n📥 Download these files from the Colab file browser (left sidebar)")
else:
    print("✗ No model files found")